## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-26-14-53-12 +0000,nypost,Investigators release new details on death of ...,https://nypost.com/2026/11/26/us-news/rock-leg...
1,2026-11-26-14-52-43 +0000,nyt,13 Dead and 13 Thought Trapped in Indonesian M...,https://www.nytimes.com/2026/11/25/world/asia/...
2,2026-11-26-14-50-27 +0000,nyt,Democrats Confront a Familiar Question: How Ar...,https://www.nytimes.com/2026/11/26/business/de...
3,2026-11-26-14-47-00 +0000,wsj,Macy’s Postpones Thanksgiving Day Parade as Po...,https://www.wsj.com/us-news/macys-postpones-it...
4,2026-11-26-14-46-00 +0000,nypost,Psycho flew from France to slaughter Nebraska ...,https://nypost.com/2026/11/26/us-news/childcar...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2719/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
29,thanksgiving,30
179,trump,29
2,new,22
185,china,15
457,nvidia,14


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
346,2026-11-25-17-24-25 +0000,wapo,"Trump proposes new tariffs on China, Europe an...",https://www.washingtonpost.com/politics/2026/1...,92
372,2026-11-25-15-58-36 +0000,nyt,Trump Administration Unveils New Tariffs for C...,https://www.nytimes.com/2026/11/25/business/tr...,89
379,2026-11-25-15-46-00 +0000,wsj,There was no evidence a Chinese tech executive...,https://www.wsj.com/politics/national-security...,85
25,2026-11-26-13-58-19 +0000,nypost,Trump kicks off Thanksgiving with late night d...,https://nypost.com/2026/11/26/us-news/presiden...,82
175,2026-11-26-04-14-04 +0000,nypost,Abdul El-Sayed rips Trump — in Thanksgiving me...,https://nypost.com/2026/11/25/us-news/abdul-el...,74


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
346,92,2026-11-25-17-24-25 +0000,wapo,"Trump proposes new tariffs on China, Europe an...",https://www.washingtonpost.com/politics/2026/1...
379,85,2026-11-25-15-46-00 +0000,wsj,There was no evidence a Chinese tech executive...,https://www.wsj.com/politics/national-security...
3,68,2026-11-26-14-47-00 +0000,wsj,Macy’s Postpones Thanksgiving Day Parade as Po...,https://www.wsj.com/us-news/macys-postpones-it...
118,65,2026-11-26-10-00-00 +0000,wsj,Democratic Sen. Chris Van Hollen wants to be S...,https://www.wsj.com/politics/policy/why-this-d...
227,59,2026-11-25-22-38-54 +0000,nypost,Elon Musk’s plan for SpaceX to launch 1 millio...,https://nypost.com/2026/11/25/business/fcc-app...
100,52,2026-11-26-10-13-24 +0000,nypost,Hong Kong marks anniversary high-rise blaze th...,https://nypost.com/2026/11/26/world-news/hong-...
126,45,2026-11-26-09-53-19 +0000,nypost,Iowa residents urged to shelter amid lead expo...,https://nypost.com/2026/11/26/us-news/iowa-res...
167,39,2026-11-26-05-04-49 +0000,nypost,Russia must avoid ‘any actions’ expanding war ...,https://nypost.com/2026/11/26/us-news/russia-m...
358,38,2026-11-25-16-43-13 +0000,nypost,"Nasdaq tumbles 4% as Nvidia leads AI sell-off,...",https://nypost.com/2026/11/25/business/nasdaq-...
53,36,2026-11-26-12-37-41 +0000,nypost,Macy’s faces staggering $1.7M loss just to blo...,https://nypost.com/2026/11/26/lifestyle/macys-...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
